# Baseline: TF-IDF + Logistic Regression / SVM

Train and evaluate the baseline ABSA model from the CLI or inline.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_processed_splits
from src.preprocess import encode_labels
from src.models.baseline import BaselineModel
from src.evaluate import compute_all_metrics, plot_confusion_matrix
from src.utils import set_seed

In [ ]:
set_seed(42)
processed_dir = ROOT / "data" / "processed"
if (processed_dir / "train.csv").exists():
    train_df, val_df, test_df = load_processed_splits(str(processed_dir))
else:
    train_df = pd.DataFrame({
        "sentence": ["Great food and service."] * 40,
        "aspect_term": ["food", "service"] * 20,
        "polarity": ["positive", "negative"] * 20,
    })
    val_df = test_df = train_df.iloc[:10]
train_df["label"] = encode_labels(train_df["polarity"])
test_df["label"] = encode_labels(test_df["polarity"])
print("Train:", len(train_df), "Test:", len(test_df))

In [ ]:
model = BaselineModel(classifier="logistic", class_weight="balanced", random_state=42)
model.fit(
    train_df["sentence"].tolist(),
    train_df["aspect_term"].tolist(),
    train_df["label"].tolist(),
)
preds = model.predict(test_df["sentence"].tolist(), test_df["aspect_term"].tolist())
metrics = compute_all_metrics(test_df["label"].tolist(), preds)
print("Metrics:", metrics)

In [ ]:
(ROOT / "results" / "figures").mkdir(parents=True, exist_ok=True)
plot_confusion_matrix(
    test_df["label"].tolist(), preds,
    save_path=ROOT / "results" / "figures" / "notebook_baseline_cm.png",
    title="Baseline Confusion Matrix",
)
print("Saved confusion matrix.")